# **Import Libraries**

In [1]:
import gc
import os
import copy
import torch
import random
import numpy as np
import pandas as pd
import seaborn as sns
import torch.nn as nn
import matplotlib.pyplot as plt

from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import MinMaxScaler

torch.set_num_threads(8)
torch.set_num_interop_threads(2)
torch.backends.mkldnn.enabled = True

In [2]:
from utils.experiment_separator import (
    separate_experiments,
    get_experiment_summary,
    print_experiment_report,
    get_experiments,
)

In [3]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# **Import Dataset**

In [4]:
PATH = "../data/data.csv"
df_raw = pd.read_csv(PATH)

# **Rename Columns**

In [5]:
df = df_raw.copy()
df.columns = ['density', 'cutting_speed', 'feed_rate', 'depth', 'axial_force', 'cutting_force']
TARGET   = 'cutting_force'
FEATURES = ['density', 'cutting_speed', 'feed_rate', 'depth']
INPUT = ['density', 'cutting_speed', 'feed_rate', 'depth', 'axial_force']

# **Separate Experiments**

In [6]:
df = separate_experiments(df)
print_experiment_report(df)
experiments = get_experiments(df)

Total unique combinations : 72
Experiment count range    : 2 – 3
Total experiments         : 198

── Experiments per combination ──
 density  cutting_speed  feed_rate  experiment_count
      10             10         10                 3
      10             10         15                 3
      10             10         20                 3
      10             16         10                 3
      10             16         15                 3
      10             16         20                 3
      10             25         10                 3
      10             25         15                 3
      10             25         20                 3
      10             40         10                 3
      10             40         15                 3
      10             40         20                 3
      10             63         10                 3
      10             63         15                 3
      10             63         20                 3
      10            

# **Preprocessing Pipeline**

## Data Cleaning

In [7]:
def apply_remove_null(df: pd.DataFrame) -> pd.DataFrame:
    return df.dropna()

## Data Scaling

In [8]:
def apply_scaling(df: pd.DataFrame, scaler: MinMaxScaler, fit: bool = False) -> pd.DataFrame:
    cols = INPUT + [TARGET]
    if fit:
        df[cols] = scaler.fit_transform(df[cols])
    else:
        df[cols] = scaler.transform(df[cols])
    return df

## Feature Engineering

In [9]:
def apply_add_feature(df: pd.DataFrame) -> pd.DataFrame:
    return df

## Data Pipeline

In [10]:
class Pipeline:
    def __init__(self):
        self.scaler = MinMaxScaler()
 
    def fit_transform(self, df: pd.DataFrame) -> pd.DataFrame:
        temp = df.copy()
        temp = apply_remove_null(temp)
        temp = apply_scaling(temp, self.scaler, fit=True)
        temp = apply_add_feature(temp)
        return temp
 
    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        temp = df.copy()
        temp = apply_remove_null(temp)
        temp = apply_scaling(temp, self.scaler, fit=False)
        temp = apply_add_feature(temp)
        return temp

# **Train/Val Split**

In [11]:
exp_keys = list(experiments.keys())
random.shuffle(exp_keys)
 
n_total = len(exp_keys)
n_train = int(n_total * 0.70)
n_val   = int(n_total * 0.15)
n_test  = n_total - n_train - n_val
 
train_keys = exp_keys[:n_train]
val_keys   = exp_keys[n_train:n_train + n_val]
test_keys  = exp_keys[n_train + n_val:]
 
print(f"\nTotal experiments : {n_total}")
print(f"Train experiments : {len(train_keys)}")
print(f"Val experiments   : {len(val_keys)}")
print(f"Test experiments  : {len(test_keys)}")


Total experiments : 198
Train experiments : 138
Val experiments   : 29
Test experiments  : 31


## Windowing

In [12]:
def experiments_to_tensor(keys, experiments, pipeline, window_size, fit=False):
    ref_df = pd.concat([experiments[k] for k in keys], ignore_index=True)
    processed = pipeline.fit_transform(ref_df) if fit else pipeline.transform(ref_df)
 
    lengths = [len(experiments[k]) for k in keys]
 
    X_windows, y_windows = [], []
    start = 0
    for length in lengths:
        exp_slice = processed.iloc[start:start+length].reset_index(drop=True)
        start += length
 
        if len(exp_slice) < window_size:
            continue
 
        input_vals  = exp_slice[INPUT].values
        target_vals = exp_slice[TARGET].values
 
        for i in range(0, len(exp_slice) - window_size + 1, STRIDE):
            X_windows.append(input_vals[i:i+window_size])
            y_windows.append(target_vals[i+window_size-1])
 
    X = torch.tensor(np.array(X_windows), dtype=torch.float32)
    y = torch.tensor(np.array(y_windows), dtype=torch.float32).unsqueeze(1)
    return X, y

# **Modeling**

In [13]:
INPUT_SIZE  = len(INPUT)
OUTPUT_SIZE = 1
# K-Fold is disabled for faster model iteration.
USE_KFOLD   = False

## hyperparameter

In [14]:
WINDOW_SIZE   = 200
STRIDE        = 10
HIDDEN_SIZE   = 128
NUM_LAYERS    = 2
EPOCHS        = 10
BATCH_SIZE    = 256
LEARNING_RATE = 5e-4
PATIENCE      = 2
MIN_DELTA     = 0.0

## Training Loop

In [15]:
def calculate_loss_in_batches(model, X, y, criterion, batch_size):
    loader = DataLoader(
        TensorDataset(X, y),
        batch_size=batch_size,
        shuffle=False
    )

    total_loss = 0.0
    total_samples = 0

    model.eval()

    with torch.no_grad():
        for X_batch, y_batch in loader:
            predictions = model(X_batch)
            loss = criterion(predictions, y_batch)

            current_batch_size = X_batch.size(0)
            total_loss += loss.item() * current_batch_size
            total_samples += current_batch_size

    return total_loss / total_samples

# Kept only as a reference. It is not called while USE_KFOLD is False.
def train_with_kfold(model_class, model_name: str, experiments: dict, train_keys: list, val_keys: list):
    kf             = KFold(n_splits=N_FOLDS, shuffle=False)
    train_key_arr  = np.array(train_keys, dtype=object)
    fold_val_losses = []
 
    print(f"\n Training {model_name} ")
 
    for fold, (fold_train_idx, fold_val_idx) in enumerate(kf.split(train_key_arr)):
        fold_train_keys = [tuple(k) for k in train_key_arr[fold_train_idx]]
        fold_val_keys   = [tuple(k) for k in train_key_arr[fold_val_idx]]
 
        model     = model_class()
        pipeline  = Pipeline()
        criterion = nn.MSELoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=model.lr)
 
        X_fold_train, y_fold_train = experiments_to_tensor(fold_train_keys, experiments, pipeline, WINDOW_SIZE, fit=True)
        X_fold_val,   y_fold_val   = experiments_to_tensor(fold_val_keys,   experiments, pipeline, WINDOW_SIZE, fit=False)
 
        train_loader = DataLoader(TensorDataset(X_fold_train, y_fold_train), batch_size=model.batch_size, shuffle=True)
 
        for epoch in range(model.epochs):
            model.train()
            train_loss = 0
            for X_batch, y_batch in train_loader:
                optimizer.zero_grad()
                loss = criterion(model(X_batch), y_batch)
                loss.backward()
                optimizer.step()
                train_loss += loss.item()
 
            val_loss = calculate_loss_in_batches(
                model,
                X_fold_val,
                y_fold_val,
                criterion,
                model.batch_size
            )
 
            print(f"  Fold {fold+1}/{N_FOLDS} | Epoch {epoch+1}/{model.epochs} | Train Loss: {train_loss/len(train_loader):.4f} | Val Loss: {val_loss:.4f}")
 
        fold_val_losses.append(val_loss)
 
        del model, optimizer, X_fold_train, y_fold_train, X_fold_val, y_fold_val, train_loader
        import gc
        gc.collect()
 
    print(f"\n  KFold Val Loss per fold (diagnostic only) : {[f'{v:.4f}' for v in fold_val_losses]}")
    print(f"  Mean Val Loss                             : {np.mean(fold_val_losses):.4f} \u00b1 {np.std(fold_val_losses):.4f}")
    print(f"\n  Retraining on full train set, checkpointing on val_keys...")
    final_model    = model_class()
    final_pipeline = Pipeline()
    criterion      = nn.MSELoss()
    optimizer      = torch.optim.Adam(final_model.parameters(), lr=final_model.lr)
 
    X_train, y_train = experiments_to_tensor(train_keys, experiments, final_pipeline, WINDOW_SIZE, fit=True)
    X_val,   y_val    = experiments_to_tensor(val_keys,   experiments, final_pipeline, WINDOW_SIZE, fit=False)
    train_loader     = DataLoader(TensorDataset(X_train, y_train), batch_size=final_model.batch_size, shuffle=True)
 
    best_val_loss   = float('inf')
    best_state_dict = None
    best_epoch      = -1
 
    for epoch in range(final_model.epochs):
        final_model.train()
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            loss = criterion(final_model(X_batch), y_batch)
            loss.backward()
            optimizer.step()
 
        val_loss = calculate_loss_in_batches(
            final_model,
            X_val,
            y_val,
            criterion,
            final_model.batch_size
        )
 
        if val_loss < best_val_loss:
            best_val_loss   = val_loss
            best_state_dict = copy.deepcopy(final_model.state_dict())
            best_epoch      = epoch + 1
 
    print(f"  Best epoch on val_keys: {best_epoch} | Best Val Loss: {best_val_loss:.4f}")
    final_model.load_state_dict(best_state_dict)
 
    return final_model, final_pipeline

def prepare_datasets(experiments, train_keys, val_keys, test_keys):
    """Create scaled windows once; LSTM, GRU, and RNN reuse them."""
    pipeline = Pipeline()
    X_train, y_train = experiments_to_tensor(train_keys, experiments, pipeline, WINDOW_SIZE, fit=True)
    X_val, y_val     = experiments_to_tensor(val_keys,   experiments, pipeline, WINDOW_SIZE, fit=False)
    X_test, y_test   = experiments_to_tensor(test_keys,  experiments, pipeline, WINDOW_SIZE, fit=False)
    print(f"Prepared tensors | train: {len(X_train):,} | val: {len(X_val):,} | test: {len(X_test):,} | stride: {STRIDE}")
    return {
        'train_dataset': TensorDataset(X_train, y_train),
        'X_val': X_val, 'y_val': y_val,
        'X_test': X_test, 'y_test': y_test,
        'pipeline': pipeline,
    }

def train(model_class, model_name: str, prepared_data: dict):
    print(f"\nTraining {model_name} (K-Fold disabled)")
    model = model_class()
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=model.lr)
    train_loader = DataLoader(prepared_data['train_dataset'], batch_size=model.batch_size, shuffle=True)

    best_val_loss = float('inf')
    best_state_dict = None
    best_epoch = -1
    epochs_without_improvement = 0

    for epoch in range(model.epochs):
        model.train()
        train_loss = 0.0
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            loss = criterion(model(X_batch), y_batch)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        val_loss = calculate_loss_in_batches(
            model, prepared_data['X_val'], prepared_data['y_val'], criterion, model.batch_size
        )
        train_loss /= len(train_loader)
        print(f"  Epoch {epoch + 1}/{model.epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

        if val_loss < best_val_loss - MIN_DELTA:
            best_val_loss = val_loss
            best_state_dict = copy.deepcopy(model.state_dict())
            best_epoch = epoch + 1
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= PATIENCE:
                print(f"  Early stopping at epoch {epoch + 1} (patience={PATIENCE})")
                break

    print(f"  Best epoch: {best_epoch} | Best Val Loss: {best_val_loss:.4f}")
    model.load_state_dict(best_state_dict)
    return model

## Validation Plot

In [16]:
def inverse_transform_target(pipeline, values):
    n_cols = len(INPUT) + 1
    dummy = np.zeros((len(values), n_cols))
    dummy[:, -1] = values
    return pipeline.scaler.inverse_transform(dummy)[:, -1]
 
 
def compute_experiment_predictions(model, key, experiments, pipeline):
    exp_df = experiments[key].copy()
    processed = pipeline.transform(exp_df)
 
    if len(processed) < WINDOW_SIZE:
        return None
    input_vals  = processed[INPUT].values
    target_vals = processed[TARGET].values
    depth_vals  = exp_df['depth'].values
    X_windows, y_windows, depth_windows = [], [], []
    for i in range(0, len(processed) - WINDOW_SIZE + 1, STRIDE):
        X_windows.append(input_vals[i:i+WINDOW_SIZE])
        y_windows.append(target_vals[i+WINDOW_SIZE-1])
        depth_windows.append(depth_vals[i+WINDOW_SIZE-1])
    X = torch.tensor(np.array(X_windows), dtype=torch.float32)
    y = torch.tensor(np.array(y_windows), dtype=torch.float32).unsqueeze(1)
 
    model.eval()
    with torch.no_grad():
        preds = model(X).squeeze().numpy()
    actuals = y.squeeze().numpy()
 
    preds_orig   = inverse_transform_target(pipeline, preds)
    actuals_orig = inverse_transform_target(pipeline, actuals)
    depth_orig   = np.array(depth_windows)
 
    return depth_orig, actuals_orig, preds_orig

## Validation Plot for Each Model

In [ ]:
def plot_validation_overview(model, model_name, eval_keys, experiments, pipeline, output_root="validation_plots"):
    out_dir = os.path.join(output_root, model_name)
    os.makedirs(out_dir, exist_ok=True)
 
    exp_numbers, mean_actuals, mean_preds = [], [], []
    all_actuals, all_preds = [], []
 
    for idx, key in enumerate(eval_keys, start=1):
        result = compute_experiment_predictions(model, key, experiments, pipeline)
        if result is None:
            continue
        _, actuals_orig, preds_orig = result
        exp_numbers.append(idx)
        mean_actuals.append(np.mean(actuals_orig))
        mean_preds.append(np.mean(preds_orig))
        all_actuals.extend(actuals_orig)
        all_preds.extend(preds_orig)
 
    rmse_of_means   = np.sqrt(mean_squared_error(mean_actuals, mean_preds))
    rmse_per_window = np.sqrt(mean_squared_error(all_actuals, all_preds))
 
    plt.figure(figsize=(10, 4))
    plt.plot(exp_numbers, mean_actuals, label="Actual",    color="steelblue", marker='o')
    plt.plot(exp_numbers, mean_preds,   label="Predicted", color="tomato", linestyle="--", marker='x')
    plt.title(f"{model_name} \u2014 Test Overview (per-window RMSE: {rmse_per_window:.4f})")
    plt.xlabel("Experiment Number")
    plt.ylabel("cutting_force (mean, unscaled)")
    plt.xticks(exp_numbers)
    plt.legend()
    plt.tight_layout()
 
    save_path = os.path.join(out_dir, f"{model_name}_test_overview.png")
    plt.savefig(save_path)
    plt.close()
    print(f"  Saved: {save_path}  |  RMSE of per-exp means: {rmse_of_means:.4f}  |  True per-window RMSE: {rmse_per_window:.4f}")

## Validation Plot for Each Experiment

In [ ]:
def plot_and_save_per_experiment(model, model_name, eval_keys, experiments, pipeline, output_root="validation_plots"):
    out_dir = os.path.join(output_root, model_name)
    os.makedirs(out_dir, exist_ok=True)
 
    for key in eval_keys:
        result = compute_experiment_predictions(model, key, experiments, pipeline)
        if result is None:
            continue
        depth_orig, actuals_orig, preds_orig = result
 
        sort_idx       = np.argsort(depth_orig)
        depth_sorted   = depth_orig[sort_idx]
        actuals_sorted = actuals_orig[sort_idx]
        preds_sorted   = preds_orig[sort_idx]
 
        rmse_exp = np.sqrt(mean_squared_error(actuals_orig, preds_orig))
 
        exp_name  = "_".join(str(k) for k in key) if isinstance(key, tuple) else str(key)
        safe_name = exp_name.replace(" ", "").replace("/", "-")
 
        plt.figure(figsize=(10, 4))
        plt.plot(depth_sorted, actuals_sorted, label="Actual", color="steelblue", marker='o')
        plt.plot(depth_sorted, preds_sorted,   label="Predicted", color="tomato", linestyle="--", marker='x')
        plt.title(f"{model_name} RMSE: {rmse_exp:.4f} (Experiment {exp_name})")
        plt.xlabel("depth")
        plt.ylabel("cutting_force")
        plt.legend()
        plt.tight_layout()
 
        save_path = os.path.join(out_dir, f"{safe_name}.png")
        plt.savefig(save_path)
        plt.close()
        print(f"  Saved: {save_path}  |  RMSE: {rmse_exp:.4f}")

## LSTM

In [19]:
class LSTM(nn.Module):
    hidden_size = HIDDEN_SIZE
    num_layers  = NUM_LAYERS
    epochs      = EPOCHS
    batch_size  = BATCH_SIZE
    lr          = LEARNING_RATE
 
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(INPUT_SIZE, self.hidden_size, self.num_layers, batch_first=True)
        self.fc   = nn.Linear(self.hidden_size, OUTPUT_SIZE)
 
    def forward(self, x):
        _, (h_n, _) = self.lstm(x)
        return self.fc(h_n[-1])

In [20]:
prepared_data = prepare_datasets(experiments, train_keys, val_keys, test_keys)
lstm_model = train(LSTM, "LSTM", prepared_data)
lstm_pipeline = prepared_data['pipeline']

 
# plot_validation_overview(lstm_model, "LSTM", test_keys, experiments, lstm_pipeline)
# plot_and_save_per_experiment(lstm_model, "LSTM", test_keys, experiments, lstm_pipeline)

Prepared tensors | train: 17,607 | val: 3,442 | test: 3,503 | stride: 10

Training LSTM (K-Fold disabled)
  Epoch 1/10 | Train Loss: 0.0071 | Val Loss: 0.0010
  Epoch 2/10 | Train Loss: 0.0008 | Val Loss: 0.0005
  Epoch 3/10 | Train Loss: 0.0005 | Val Loss: 0.0005
  Epoch 4/10 | Train Loss: 0.0004 | Val Loss: 0.0004
  Epoch 5/10 | Train Loss: 0.0003 | Val Loss: 0.0003
  Epoch 6/10 | Train Loss: 0.0002 | Val Loss: 0.0002
  Epoch 7/10 | Train Loss: 0.0002 | Val Loss: 0.0001
  Epoch 8/10 | Train Loss: 0.0002 | Val Loss: 0.0001
  Epoch 9/10 | Train Loss: 0.0001 | Val Loss: 0.0001
  Epoch 10/10 | Train Loss: 0.0002 | Val Loss: 0.0002
  Best epoch: 9 | Best Val Loss: 0.0001


In [21]:
# import joblib

# checkpoint = torch.load(
#     "saved_models/lstm_checkpoint.pth",
#     map_location="cpu"
# )

# lstm_model = LSTM()

# lstm_model.load_state_dict(
#     checkpoint["model_state_dict"]
# )

# lstm_model.eval()

# lstm_pipeline = Pipeline()

# lstm_pipeline.scaler = joblib.load(
#     "saved_models/lstm_scaler.pkl"
# )

# print("LSTM berhasil dimuat kembali.")

In [22]:
# print(type(lstm_model))
# print(type(lstm_pipeline))

In [23]:
# print(type(lstm_model))
# print(type(lstm_pipeline))

# X_test_lstm, y_test_lstm = experiments_to_tensor(
#     test_keys,
#     experiments,
#     lstm_pipeline,
#     WINDOW_SIZE,
#     fit=False
# )

In [24]:
# import os
# import joblib
# import torch

# os.makedirs("saved_models", exist_ok=True)

# torch.save(
#     {
#         "model_state_dict": lstm_model.state_dict(),
#         "input_size": INPUT_SIZE,
#         "hidden_size": HIDDEN_SIZE,
#         "num_layers": NUM_LAYERS,
#         "output_size": OUTPUT_SIZE,
#         "window_size": WINDOW_SIZE,
#         "input_columns": INPUT,
#         "target_column": TARGET
#     },
#     "saved_models/lstm_checkpoint.pth"
# )

# joblib.dump(
#     lstm_pipeline.scaler,
#     "saved_models/lstm_scaler.pkl"
# )

# print("Model saved:", os.path.exists("saved_models/lstm_checkpoint.pth"))
# print("Scaler saved:", os.path.exists("saved_models/lstm_scaler.pkl"))

## GRU

In [25]:
class GRU(nn.Module):
    hidden_size = HIDDEN_SIZE
    num_layers  = NUM_LAYERS
    epochs      = EPOCHS
    batch_size  = BATCH_SIZE
    lr          = LEARNING_RATE
 
    def __init__(self):
        super().__init__()
        self.gru = nn.GRU(INPUT_SIZE, self.hidden_size, self.num_layers, batch_first=True)
        self.fc  = nn.Linear(self.hidden_size, OUTPUT_SIZE)
 
    def forward(self, x):
        _, h_n = self.gru(x)
        return self.fc(h_n[-1])

In [26]:
gru_model = train(GRU, "GRU", prepared_data)
gru_pipeline = prepared_data['pipeline']
 
# plot_validation_overview(gru_model, "GRU", test_keys, experiments, gru_pipeline)
# plot_and_save_per_experiment(gru_model, "GRU", test_keys, experiments, gru_pipeline)


Training GRU (K-Fold disabled)


  Epoch 1/10 | Train Loss: 0.0092 | Val Loss: 0.0042
  Epoch 2/10 | Train Loss: 0.0025 | Val Loss: 0.0013
  Epoch 3/10 | Train Loss: 0.0007 | Val Loss: 0.0005
  Epoch 4/10 | Train Loss: 0.0006 | Val Loss: 0.0005
  Epoch 5/10 | Train Loss: 0.0004 | Val Loss: 0.0004
  Epoch 6/10 | Train Loss: 0.0004 | Val Loss: 0.0004
  Epoch 7/10 | Train Loss: 0.0005 | Val Loss: 0.0004
  Epoch 8/10 | Train Loss: 0.0004 | Val Loss: 0.0003
  Epoch 9/10 | Train Loss: 0.0003 | Val Loss: 0.0002
  Epoch 10/10 | Train Loss: 0.0002 | Val Loss: 0.0002
  Best epoch: 10 | Best Val Loss: 0.0002


## RNN

In [27]:
class RNN(nn.Module):
    hidden_size = HIDDEN_SIZE
    num_layers  = NUM_LAYERS
    epochs      = EPOCHS
    batch_size  = BATCH_SIZE
    lr          = LEARNING_RATE
 
    def __init__(self):
        super().__init__()
        self.rnn = nn.RNN(INPUT_SIZE, self.hidden_size, self.num_layers, batch_first=True)
        self.fc  = nn.Linear(self.hidden_size, OUTPUT_SIZE)
 
    def forward(self, x):
        _, h_n = self.rnn(x)
        return self.fc(h_n[-1])

In [28]:
rnn_model = train(RNN, "RNN", prepared_data)
rnn_pipeline = prepared_data['pipeline']
 
# plot_validation_overview(rnn_model, "RNN", test_keys, experiments, rnn_pipeline)
# plot_and_save_per_experiment(rnn_model, "RNN", test_keys, experiments, rnn_pipeline)


Training RNN (K-Fold disabled)
  Epoch 1/10 | Train Loss: 0.0078 | Val Loss: 0.0046
  Epoch 2/10 | Train Loss: 0.0047 | Val Loss: 0.0045
  Epoch 3/10 | Train Loss: 0.0046 | Val Loss: 0.0042
  Epoch 4/10 | Train Loss: 0.0044 | Val Loss: 0.0037
  Epoch 5/10 | Train Loss: 0.0016 | Val Loss: 0.0006
  Epoch 6/10 | Train Loss: 0.0006 | Val Loss: 0.0006
  Epoch 7/10 | Train Loss: 0.0006 | Val Loss: 0.0006
  Early stopping at epoch 7 (patience=2)
  Best epoch: 5 | Best Val Loss: 0.0006


# **Evaluation**

In [29]:
def evaluate(model, model_name, X_test, y_test, pipeline):
    test_loader = DataLoader(
        TensorDataset(X_test, y_test),
        batch_size=model.batch_size,
        shuffle=False
    )

    all_preds = []
    all_actuals = []

    model.eval()

    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            batch_preds = model(X_batch)

            all_preds.append(batch_preds.cpu())
            all_actuals.append(y_batch.cpu())

    preds = torch.cat(all_preds, dim=0).squeeze().numpy()
    actuals = torch.cat(all_actuals, dim=0).squeeze().numpy()

    n_cols = len(INPUT) + 1

    dummy_preds = np.zeros((len(preds), n_cols))
    dummy_preds[:, -1] = preds
    preds_orig = pipeline.scaler.inverse_transform(dummy_preds)[:, -1]

    dummy_actuals = np.zeros((len(actuals), n_cols))
    dummy_actuals[:, -1] = actuals
    actuals_orig = pipeline.scaler.inverse_transform(dummy_actuals)[:, -1]

    mae  = mean_absolute_error(actuals_orig, preds_orig)
    mse  = mean_squared_error(actuals_orig, preds_orig)
    rmse = np.sqrt(mse)
    std  = np.std(actuals_orig)

    print(f" {model_name} (TEST) ")
    print(f"  MAE  : {mae:.4f} ({mae/std*100:.2f}% of std)")
    print(f"  MSE  : {mse:.4f}")
    print(f"  RMSE : {rmse:.4f} ({rmse/std*100:.2f}% of std)")
    print(f"  STD  : {std:.4f}")
    print()
 
evaluate(lstm_model, "LSTM", prepared_data['X_test'], prepared_data['y_test'], lstm_pipeline)
evaluate(gru_model,  "GRU",  prepared_data['X_test'], prepared_data['y_test'], gru_pipeline)
evaluate(rnn_model,  "RNN",  prepared_data['X_test'], prepared_data['y_test'], rnn_pipeline)

del prepared_data

del lstm_model, lstm_pipeline
del gru_model, gru_pipeline
del rnn_model, rnn_pipeline

gc.collect()

 LSTM (TEST) 
  MAE  : 16.0096 (6.79% of std)
  MSE  : 601.7548
  RMSE : 24.5307 (10.41% of std)
  STD  : 235.7551

 GRU (TEST) 
  MAE  : 16.6776 (7.07% of std)
  MSE  : 663.8620
  RMSE : 25.7655 (10.93% of std)
  STD  : 235.7551

 RNN (TEST) 
  MAE  : 32.0713 (13.60% of std)
  MSE  : 1970.5622
  RMSE : 44.3910 (18.83% of std)
  STD  : 235.7551



0